# California housing 

## Regression - förutsäga bostadsvärde

I denna notebook analyseras ett dataset som beskriver bostadsvärden i Kalifornien. 
Målet är att utveckla en modell för att förutsäga medianvärdet på bostäder i olika områden. Modellen kan sen användas som beslutsstöd vid exempelvis investeringar eller planering inom bostadsmarknaden för att ge en uppskattning av förväntade bostadspriser.

Strukturen är följande:
- Dataförståelse & EDA - Undersöka datasetet, visuell analys av variabler
- Split + preprocessing - förbered datan för modellträning
- Modellering - träna och jämföra flera modeller
- Optimera en modell - optimera den bästa modellen
- Utvärdering testdata - den optimerade modellen utvärderas på testdata
- Resultat och rekommendation - resultaten sammanfattas och en rekommendation ges

## Dataförståelse och EDA

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
df = pd.read_csv("housing.csv")

display(df.head())

print()
df.info()

print("\n*** Sammanfattning datatset ***")
display(df.describe().T)

In [ ]:
print("--Datatyper i datasetet--\n")
print(df.dtypes)

### Saknade värden

- För att hantera saknade värden så imputerar vi dem med medianen, för att minimera påverkan av outliers. Här används median istället för medelvärde då den är snedfördelad, endast ```total_bedrooms``` innehåller saknade värden.

In [ ]:
print("Shape:", df.shape)
print("\n*** Saknade värden per kolumn ***\n")
print(df.isna().sum())

### Feature engineering

In [ ]:
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df["median_house_value"], kde=True)
plt.title("Distribution av median bostadsvärden")
plt.xlabel("Median bostadsvärde (USD)")
plt.ylabel("Antal distrikt")
plt.show()

plt.figure(figsize=(8,6))
plt.scatter(df["longitude"], df["latitude"],
            c=df["median_house_value"],
            cmap="viridis", alpha=0.4)
plt.colorbar(label="Median House Value")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geografisk spridning av bostadsvärden")
plt.show()

plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", annot=False)
plt.title("Korrelationsmatris")
plt.show()

### Figurer

Den översta figuren, histplot, kan man avläsa är högerskev med en tydlig övre gräns (runt 500 000), vilket kan påverka modellens precision vid höga värden.

Figur 2, geografisk spridning av husvärden, här kan man se att geografi påverkar värdet, där priserna är högre nära kusten.

Figur 3, korrelationsmatris, visar en positiv korrelation mellan medianinkomst och bostadsvärde, och man kan även se här att geografi påverkar värdet. 

## Split + preprocessing

In [ ]:
target_col = "median_house_value"
X = df.drop(columns=[target_col])
y = df[target_col]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

numerical_features = X_train.select_dtypes(include=["int64","float64"]).columns
categorical_features = ["ocean_proximity"]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", y_train.mean().round(3), "Test positive rate:", y_test.mean().round(3))

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

## Modellering

Här används Dummyregressor som baseline, och Linear Regression och Random Forest för de andra modellerna att träna datan på


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def cv_model(model, X, y):
    scores = cross_val_score(
        model, 
        X,
        y,
        scoring="neg_root_mean_squared_error",
        cv=kf
    )
    return -scores.mean()

### Baseline

In [ ]:
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

baseline_scores = cv_model(baseline_model, X_train, y_train)
print(baseline_scores.mean())

### Linear regression

In [ ]:
linear_reg_pl = Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())])

lin_reg_scores = cv_model(linear_reg_pl, X_train, y_train)

print(lin_reg_scores)

### Random forest

In [ ]:
rand_forest_pl = Pipeline([
    ("preprocessor", preprocessor), 
    ("model", RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE))
    ])

rand_forest_scores = cv_model(rand_forest_pl, X_train, y_train)

print(rand_forest_scores.mean())

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline", "Linear Regression", "Random Forest"],
    "CV_RMSE": [
        baseline_scores,
        lin_reg_scores,
        rand_forest_scores
    ]
})

results.sort_values("CV_RMSE")

RMSE valdes då den straffar stora fel mer. Inom fastighetsvärdering är stora fel mer problematiska än små. 

Man vill ha lägre RMSE, vilket man kan se att Random Forest har och presterade därmed bäst av alla modellerna. 

## Optimera en modell

Här valdes Random Forest för optimering då den presterade bäst under utvärderingen (CV)

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    rand_forest_pl,
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

In [ ]:
grid_search.fit(X_train, y_train)
print("Bäst CV score:", -grid_search.best_score_)
print("Bäst params:", grid_search.best_params_)

In [ ]:
def evaluate_regression(y_true, y_pred) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }

Hyperparameteroptimering gjordes med GridSearchCV och en 5-fold cross-validation.

Valde att optimera följande parametrar:
- Antalet träd ```n_estimators```
- Trädens djup ```max_depth```
- Hur lätt träden delar upp datan ```min_samples_split```
- Hur stora bladnoderna får vara ```min_samples_leaf```

Metric att optimera mot är ```neg_root_mean_squared_error```.

## Utvärdering testdata

In [ ]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

test_results = evaluate_regression(y_test, y_pred)

results_df = pd.DataFrame([test_results])
results_df.insert(0, "Model", "Random Forest (Tuned)")

results_df

In [ ]:
baseline_model.fit(X_train, y_train)
linear_reg_pl.fit(X_train, y_train)
rand_forest_pl.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)
lin_pred = linear_reg_pl.predict(X_test)
rf_pred = rand_forest_pl.predict(X_test)
best_pred = best_model.predict(X_test)

baseline_results = evaluate_regression(y_test, baseline_pred)
lin_results = evaluate_regression(y_test, lin_pred)
rf_results = evaluate_regression(y_test, rf_pred)
best_results = evaluate_regression(y_test, best_pred)

results_df = pd.DataFrame([
    {"Model": "Baseline (Dummy)", **baseline_results},
    {"Model": "Linear Regression", **lin_results},
    {"Model": "Random Forest", **rf_results},
    {"Model": "Random Forest (Tuned)", **best_results},
])

results_df

## Resultat

Valde tre modeller att träna datan på: Dummyregressor (baseline), Linear Regression och Forest Regressor. 

Modellerna implementerades i en gemensam pipeline med preprocessing för att undvika data leakage.

Modellerna jämfördes sedan med en 5-fold cross-validation och RMSE som utvärderingsmetod.
Random Forest hade lägst RMSE och valdes som den slutliga modellen för testdatan.

Valet föll till RMSE då den straffar stora prediktionsfel mer, vilket är passande där stora värderingsfel innebär en hög risk.

## Rekommendation

Som man kan se från tabellen lite högre upp, så presterade den optimerade Random Forest tabellen bäst på testdatan, med lägst RMSE och MAE, samt högst R^2.
Med det menas att modellen ger mer exakta prediktioner än de andra modellerna. 

En annan fördel med Random Forest är att den kan hantera icke-linjära interaktioner mellan variabler, vilket passar för bostadsdata av denna typ.

Med andra ord så rekommenderas den optimerade Random Forest modellen som beslutsstöd för att uppskatta bostadsvärden i olika områden.